# Set up and update the ingestion watermark

Attach `lh_meridian_hr` as the default lakehouse. In Fabric, mark Cell 2 as the parameter cell so the pipeline can override the values when advancing the watermark. Setup mode creates and seeds missing state; update mode advances the selected pipeline watermark.

## Parameters (mark this as the parameter cell)

**Summary.** Declares the three values the pipeline overrides at run time: the `mode`, which pipeline the watermark belongs to, and the timestamp to seed or advance to.

<details>
<summary>Line-by-line details</summary>

- `mode = "setup"` — `setup` creates/seeds state only when missing; `update` advances the watermark. The pipeline overrides this per activity.
- `pipeline_name = "workforce_events"` — the key that identifies this pipeline's row in the watermark table.
- `watermark_timestamp = "2020-12-01 00:00:00"` — the value to seed (setup) or store (update); a deliberately old default so the first load picks up everything.

</details>

In [ ]:
mode = "setup"
pipeline_name = "workforce_events"
watermark_timestamp = "2020-12-01 00:00:00"

## Create, seed, or advance the watermark

**Summary.** Validates the parameters, ensures the `bronze.ingestion_watermark` control table exists, then MERGEs a single row for this pipeline — inserting on `setup` and updating on `update`.

<details>
<summary>Line-by-line details</summary>

- `from datetime import datetime` — used to parse and validate the supplied timestamp.
- The two `if ... raise ValueError` guards reject an unknown `mode` and an empty `pipeline_name`.
- `datetime.strptime(watermark_timestamp, "%Y-%m-%d %H:%M:%S")` — parses the timestamp; the following `if parsed_watermark.day != 1` enforces the first-of-month convention.
- `CREATE SCHEMA IF NOT EXISTS bronze` and `CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (...)` — idempotently create the control table (`pipeline_name`, `watermark_timestamp`, `updated_at`).
- `spark.createDataFrame([...])` + `createOrReplaceTempView("watermark_input")` — build a one-row source for the MERGE.
- `if mode == "update"` — the MERGE updates `watermark_timestamp`/`updated_at` when the row exists **and** inserts if it does not.
- `else` (setup) — the MERGE only inserts when the row is missing, so an existing watermark is never overwritten during setup.
- The final `SELECT ... show(...)` prints the current watermark rows for confirmation.

</details>

In [ ]:
from datetime import datetime

if mode not in {"setup", "update"}:
    raise ValueError("mode must be 'setup' or 'update'")
if not pipeline_name.strip():
    raise ValueError("pipeline_name must not be empty")

parsed_watermark = datetime.strptime(watermark_timestamp, "%Y-%m-%d %H:%M:%S")
if parsed_watermark.day != 1:
    raise ValueError("watermark_timestamp must be the first day of a month")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (
    pipeline_name STRING,
    watermark_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

watermark_input = spark.createDataFrame(
    [(pipeline_name, parsed_watermark)],
    "pipeline_name STRING, watermark_timestamp TIMESTAMP",
)
watermark_input.createOrReplaceTempView("watermark_input")

if mode == "update":
    spark.sql("""
    MERGE INTO bronze.ingestion_watermark AS target
    USING watermark_input AS source
    ON target.pipeline_name = source.pipeline_name
    WHEN MATCHED THEN UPDATE SET
        target.watermark_timestamp = source.watermark_timestamp,
        target.updated_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
        VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
    """)
else:
    spark.sql("""
    MERGE INTO bronze.ingestion_watermark AS target
    USING watermark_input AS source
    ON target.pipeline_name = source.pipeline_name
    WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
        VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
    """)

spark.sql("SELECT * FROM bronze.ingestion_watermark ORDER BY pipeline_name").show(truncate=False)